# Rosie’s Halloween · Bakery 正式聲線品質測試

只使用已核定的六角色設定。此階段只產生 **Bakery Rosie/Ghost**，不生成全劇118句；不呼叫付費API、不啟用麥克風。

1. 選「執行階段 → 變更執行階段類型 → T4 GPU」。免費GPU不保證可取得，沒有就停止，不自動改付費。
2. 依序執行安裝、上傳 **bakery-colab-inputs.zip**、檢查參考音與工作清單。這個ZIP含你提供的參考音，請只在自己的Colab上傳，不要公開分享輸入包。
3. 預設是dry-run，只列出6句。確認後將 RUN_GENERATION 改成 True，才真的生成6句。
4. 下載輸出ZIP、解壓縮，到獨立測試頁選取 generated-manifest.json 及 audio 內的WAV。
5. 6句品質確認後，才選 bakery_complete 與 BAKERY_ONLY_53。正式遊戲從不自動修改。

Ghost主要參考：ghost_qwen_boy_v1.wav；Pirate主要參考：pirate_A_captain.wav。clone_test只供播放比較。Rosie使用已保存的qwen_rosie_r2，不重新設計聲音。

In [ ]:
import sys, subprocess, os
os.environ['HF_HUB_DISABLE_IMPLICIT_TOKEN']='1'
os.environ['HF_HUB_DISABLE_TELEMETRY']='1'
subprocess.check_call(['apt-get','-qq','update'])
subprocess.check_call(['apt-get','-qq','install','-y','espeak-ng','sox','libsndfile1'])
subprocess.check_call([sys.executable,'-m','pip','install','-q','qwen-tts==0.1.1','transformers==4.57.3','accelerate==1.12.0','kokoro==0.9.4','soundfile','huggingface-hub>=0.34,<1'])
print('套件安裝完成。若Colab提示套件已載入，請重新啟動執行階段，再從上傳格繼續。')

In [ ]:
from google.colab import files
from pathlib import Path
import io, zipfile, json, hashlib
WORK=Path('/content/rosie_bakery_work')
WORK.mkdir(parents=True,exist_ok=True)
uploaded=files.upload()  # 選取本機 bakery-colab-inputs.zip
if len(uploaded)!=1: raise ValueError('請只上傳一個 bakery-colab-inputs.zip')
raw=next(iter(uploaded.values()))
with zipfile.ZipFile(io.BytesIO(raw)) as archive:
    if sum(i.file_size for i in archive.infolist())>100_000_000: raise ValueError('輸入包過大')
    for info in archive.infolist():
        target=(WORK/info.filename).resolve()
        if not target.is_relative_to(WORK.resolve()): raise ValueError('不安全的ZIP路徑')
        if info.is_dir(): continue
        target.parent.mkdir(parents=True,exist_ok=True)
        target.write_bytes(archive.read(info))
print('已解壓至',WORK)
import importlib.util
spec=importlib.util.spec_from_file_location('bakery_runner',WORK/'colab_runner.py')
runner=importlib.util.module_from_spec(spec);spec.loader.exec_module(runner)
manifest,jobs=runner.plan(WORK,'bakery_smoke')
print('engine來源',manifest['source'])
print('主要參考音、SHA256、逐字稿已驗證；測試音不會用作主要參考。')

In [ ]:
from IPython.display import Audio, display
for role in ['rosie','ghost','pirate']:
    c=manifest['characters'][role]
    print(role,'主要參考：',c['reference']['file'])
    print('ref_text =',repr(c['reference']['ref_text']))
    display(Audio(filename=str(WORK/'references'/c['reference']['file'])))
    test=c.get('consistencyReference')
    if test:
        print('以下只作聲線一致性比較，不輸入生成模型：',test['file'])
        display(Audio(filename=str(WORK/'references'/test['file'])))

In [ ]:
# 預設只整理清單，不執行推論。第一次請保留 bakery_smoke。
RUN_GENERATION=False
JOB_SCOPE='bakery_smoke'  # 6句；確認後才可改為 bakery_complete（53個音檔）
BAKERY_CONFIRMATION=''   # 完整Bakery需明確填入 BAKERY_ONLY_53
OUTPUT=Path('/content/rosie_bakery_output')
planned=runner.generate(WORK,OUTPUT,scope=JOB_SCOPE,run=False)
for job in planned['jobs']:
    print(job['role'],job['voiceId'],job['assetId'],repr(job['text']))
print('目前只列清單，未生成音檔。')

In [ ]:
# 真正生成；GPU時間及速度依免費Colab資源而異。
if RUN_GENERATION:
    generated=runner.generate(WORK,OUTPUT,scope=JOB_SCOPE,run=True,bakery_confirmation=BAKERY_CONFIRMATION)
    print('本次輸出',len(generated['assets']),'個音檔，全部仍等待遊戲內品質驗收。')
else:
    print('RUN_GENERATION=False：未載入模型、未生成音檔。要試做6句時，修改上一格再執行本格。')

In [ ]:
# 發生中斷仍可執行這格：下載已完成部分，同一Colab執行階段可重跑以保留進度；新執行階段不會自動恢復。
from IPython.display import Audio, display
if (OUTPUT/'generated-manifest.json').exists():
    generated=json.loads((OUTPUT/'generated-manifest.json').read_text(encoding='utf-8'))
    for asset in generated['assets'].values():
        print(asset['role'],asset['voiceId'],asset['text'],asset['review'])
        display(Audio(filename=str(OUTPUT/asset['src'])))
    import shutil
    bundle=shutil.make_archive('/content/bakery-generated-review','zip',OUTPUT)
    files.download(bundle)
else:
    print('尚無生成結果可下載，請先完成6句測試。')

## 驗收後的下一步

- 先比較Ghost主要參考、clone_test與新句，確認男孩感、頑皮感、美式發音及字尾；Rosie不得改用新的VoiceDesign。
- 6句確認後才執行完整Bakery 53個音檔；保留逐句重試與數字變體。
- 接著匯入獨立Bakery測試頁，確認字幕、嘴型起停、音樂ducking、暫停/重播/跳過/返回。
- 本輪僅能提供RMS音量嘴型，沒有phoneme時間對齊，不能稱精準音素嘴型。
- 原始參考音的來源與額外授權證明由提供者保留；輸入ZIP不會發布到公開網站。
- 本Notebook尚未在你的Colab GPU實際執行；交付檢查為Notebook結構/程式語法/工作規劃與模型呼叫介面測試。免費GPU限制見 [Colab FAQ](https://research.google.com/colaboratory/faq.html)，模型API見 [Qwen官方文件](https://github.com/QwenLM/Qwen3-TTS)。